# Arabic fake news

**1-Importing Libraries**

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install --upgrade tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.4/620.4 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 75.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 65.2 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.0
    Uninstalling tensorboard-2.17.0:
      Successfully uninstalled tensorboard-2.17.0
  Attempting uninstall: keras
    Found existing installation: keras 3.4.1
   

In [5]:
import numpy as np
import pandas as pd
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
from tqdm import tqdm
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
#from qalsadi.lemmatizer import Lemmatizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, Concatenate, BatchNormalization, Bidirectional, LSTM, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Flatten, Reshape)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
#import swifter
from transformers import TFDistilBertForSequenceClassification, DistilBertTokenizer
from transformers import AdamWeightDecay
import torch
import torch
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim import AdamW
from transformers import TrainingArguments, Trainer

**2-Load and Inspect Dataset**

In [6]:
df2 = pd.read_csv(r"/kaggle/input/arabic-fake-news-processed-data/arabic_fake_news_processed.csv")
print("Size of the data", df2.shape)
df2.head()

Size of the data (100000, 3)


,label,Article_content,cleaned_text
0,credible,بوينغ توصي شركات الطيران بوقف رحلات طائرات 777...,بوينغ وصى شركة طير وقف رحلة طاءرات حت اصدار تو...
1,credible,الدحيل والريان في قمة مباريات دوري اليد متابعة...,الدحيل ري قمه مباريات دوري يد متابع رمض مسعد ا...
2,credible,حسن بناجح.. في الحاجة إلى “اللّجام” لإكمال عُد...,حسن ناجح الحاجه ال لجام لاكمال عد حمار برلمان ...
3,credible,باحث جامعي: مهنية الأجهزة الأمنية المغربية أكس...,باحث جامع مهنة الاجهزه الامنيه المغربيه كسب اش...
4,credible,دعوى قضائية جديدة ترفع ضد ترامب رفع نائب ديموق...,دعوة قضاء جديد ترفع ضد ترامب رفع ناءب ديموقراط...


**3-Resize & Reshape for data**

In [ ]:
'''df2 = df2.drop(['source_num','publishing_date'], axis=1, errors='ignore')
df2.dropna(subset=['title', 'text', 'label'], inplace=True)
df2['Article_content'] = df2['title'] + " " + df2['text']
df2.dropna(subset=['label', 'Article_content'], inplace=True)
df2 = df2.drop(['title','text'], axis=1, errors='ignore')
df2.head()'''

In [ ]:
'''print("\n" + "="*50)
print("Data Distribution Before Sampling:")
print("="*50)
print(df2['label'].value_counts())
print(f"\nTotal samples: {len(df2)}")'''

In [ ]:
'''print("\n" + "="*50)
print("Sampling 50,000 from each class...")
print("="*50)
unique_classes = df2['label'].unique()
print(f"Unique classes: {unique_classes}")

sampled_dfs = []
for class_label in unique_classes:
    class_df = df2[df2['label'] == class_label].copy()

    if len(class_df) > 50000:
        sampled_class = class_df.sample(n=50000, random_state=42, replace=False)
        print(f"Class '{class_label}': Sampled 50,000 from {len(class_df)} samples")
    else:
        sampled_class = class_df
        print(f"Class '{class_label}': Using all {len(class_df)} samples (less than 50,000)")

    sampled_dfs.append(sampled_class)

df_sampled = pd.concat(sampled_dfs, ignore_index=True)

print("\n" + "="*50)
print("Data Distribution After Sampling:")
print("="*50)
print(df_sampled['label'].value_counts())
print(f"\nTotal samples after sampling: {len(df_sampled)}")

df2 = df_sampled'''

In [7]:
y = df2['label'].values
X = df2['Article_content'].values

le = LabelEncoder()
y_encoded = le.fit_transform(y)

**4-Data preprocessing**

In [ ]:
'''arabic_stopwords = set(stopwords.words('arabic'))
negation_words = {"لن", "لا", "لم", "ليس", "ما", "غير", "بدون", "مو", "مش"}
important_words = {"بين","على","قبل","بعد","حول","حتى","أمام","خلال","ضد","إلى","منذ","وراء","تحت","وسط","كما","حيث","إذا","حين","أما","بسبب","نتيجة"}
extra_remove = {"انا","هو","هي","هم","نحن","كما","كان","كانت","يكون","قد","ثم","سوف","كل","أي","أيضا","ولكن","مع","لدى","لدي","هذا","هذه","ذلك","تلك","هناك","هنا","الى","في","عن","من","على","و","يا"}
arabic_stopwords = (arabic_stopwords | extra_remove) - (important_words | negation_words)'''

In [ ]:
'''lemmer = Lemmatizer()'''

In [ ]:
'''arabic_diacritics = re.compile("""
                             ّ    | # Tashdid
                             َ    | # Fatha
                             ً    | # Tanwin Fath
                             ُ    | # Damma
                             ٌ    | # Tanwin Damm
                             ِ    | # Kasra
                             ٍ    | # Tanwin Kasr
                             ْ    | # Sukun
                             ـ     # Tatwil/Kashida
                         """, re.VERBOSE)'''

In [ ]:
'''def normalize_arabic(text):
    text = re.sub("[إأٱآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)
    return text

def preprocess_stem(text):
    if not isinstance(text,str) or len(text.strip())==0:
        return ""

    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(arabic_diacritics, '', text)
    text = normalize_arabic(text)
    tokens = text.split()
    tokens = [lemmer.lemmatize(t) for t in tokens if t not in arabic_stopwords and len(t) > 1]
    return " ".join(tokens)'''

In [ ]:
'''print("Processing texts with strong Lemmatizing... This may take a while")
df2['cleaned_text'] = df2['Article_content'].swifter.progress_bar(True).apply(preprocess_stem)

df2 = df2[df2['cleaned_text'].str.strip() != ""]
print("Rows after preprocessing:", len(df2))'''

In [ ]:
'''df2.to_csv('arabic_fake_news_processed.csv', index=False)
print("The processed data was saved in a file'arabic_fake_news_processed.csv'")'''

In [8]:
print("Length of the first cleaned text:", len(df2['cleaned_text'].iloc[0].split()))
print("\nSample of the first cleaned text:")
print(df2['cleaned_text'].iloc[0][:500])

Length of the first cleaned text: 235

Sample of the first cleaned text:
بوينغ وصى شركة طير وقف رحلة طاءرات حت اصدار توجيه بعد فحص واشنطن قن اوصت شرك بوينغ يوم شركة طير وقف رحلة طاءرات بوينغ المماثله لطاءره شرك يونايتد ايرلاينز احترق محرك مءخرا ال ان صدر اداره طير الاتحاديه الامريكيه توجيه بعد فحص علق يابان استخدام الطاءرات حين دارس اجراءات اخري قال بوينغ انها وصى تعليق عمل طاءره طراز الخدمه مخزن التي عمل محرك بر ندا تن حت تحدد اداره طير الاتحاديه بروتوكول فحص مناسب وكانت اداره طير الاتحاديه الامريكيه ذكر انها صدر تعليم طارءه تطلب عملة تفتيش مكثف لطاءرات بوينغ عمل مح


In [9]:
le = LabelEncoder()
df2['Label'] = le.fit_transform(df2['label'])
X = df2['cleaned_text'].values
y = df2['Label'].values

**5-Split the data into training, validation, and test sets**

In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(len(X_train), len(X_val), len(X_test))

70000 15000 15000


In [11]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(class_weight='balanced',classes=np.unique(y_train),y=y_train)
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: 1.0, 1: 1.0}


In [12]:
for label, weight in class_weights.items():
    label_name = le.inverse_transform([label])[0]
    count = np.sum(y_train == label)
    print(f"   {label_name} ({label}): {count:,} Sample - weight: {weight:.3f}")

   credible (0): 35,000 Sample - weight: 1.000
   not credible (1): 35,000 Sample - weight: 1.000


**6-Tokenization and word embadding**

In [ ]:
vocab_size = 20000
embedding_dim = 128
max_length = 512
oov_tok = "<OOV>"
num_epochs = 30

In [ ]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

In [ ]:
train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_length, padding='post')
val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=max_length, padding='post')
test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_length, padding='post')
print("Final data shape:", train_seq.shape)

**7-TF-IDF**

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

In [ ]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF training shape: {X_train_tfidf.shape}")
print(f"Number of features: {X_train_tfidf.shape[1]}")

**8-EvaluEvaluation Model**

In [ ]:
def evaluate_model(y_true, y_pred, y_pred_prob=None, model_name="Model"):

    print(f"\n{'='*50}")
    print(f"Evaluate {model_name}")
    print(f"{'='*50}")
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f" Accuracy: {accuracy:.4f}")
    print(f" Precision: {precision:.4f}")
    print(f" Recall: {recall:.4f}")
    print(f" F1-Score: {f1:.4f}")
    print(f"\n Classification report:")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicate')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

**9-Models Building and Training**

1-Bilstm

In [ ]:
def build_bilstm_model(vocab_size, embedding_dim, max_length):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        Dropout(0.2),
        Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
        Bidirectional(LSTM(64, dropout=0.3)),
        Dense(128, activation='relu'),
        Dropout(0.4),
        BatchNormalization(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    return model


In [ ]:
bilstm_model = build_bilstm_model(vocab_size, embedding_dim, max_length)
bilstm_model.build(input_shape=(None, max_length))
bilstm_model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
bilstm_model.summary()

In [ ]:
bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', restore_best_weights=True,patience=3, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

In [ ]:
start_time = time.time()
bilstm_history = bilstm_model.fit(
    train_seq, y_train,
    validation_data=(val_seq, y_val),
    epochs=15,
    batch_size=128,
    class_weight=class_weights,
    callbacks=bilstm_callbacks,
    verbose=1
)
bilstm_time = time.time() - start_time

In [ ]:
bilstm_pred_prob = bilstm_model.predict(test_seq, verbose=0)
bilstm_pred = (bilstm_pred_prob > 0.5).astype(int).flatten()
bilstm_results = evaluate_model(y_test, bilstm_pred, bilstm_pred_prob, "BiLSTM")
bilstm_results['training_time'] = bilstm_time

2-CNN-BILSTM

In [ ]:
def build_cnn_bilstm_model(vocab_size, embedding_dim, max_length):
    inputs = Input(shape=(max_length,))
    embedding = Embedding(vocab_size + 1, embedding_dim, input_length=max_length, mask_zero=True)(inputs)
    embedding = Dropout(0.2)(embedding)
    
    conv1 = Conv1D(128, 3, activation='relu', padding='same')(embedding)
    conv1 = MaxPooling1D(2)(conv1)
    conv1 = Dropout(0.2)(conv1)
    conv2 = Conv1D(64, 5, activation='relu', padding='same')(conv1)
    conv2 = MaxPooling1D(2)(conv2)
    conv2 = Dropout(0.2)(conv2)
    
    bilstm1 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(conv2)
    bilstm2 = Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.1))(bilstm1)
    
    dense = Dense(64, activation='relu')(bilstm2)
    dense = BatchNormalization()(dense)
    dense = Dropout(0.3)(dense)
    
    dense = Dense(32, activation='relu')(dense)
    dense = Dropout(0.3)(dense)
    
    outputs = Dense(1, activation='sigmoid')(dense)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
cnn_bilstm_model = build_cnn_bilstm_model(vocab_size, embedding_dim, max_length)
cnn_bilstm_model.build(input_shape=(None, max_length))
cnn_bilstm_model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
cnn_bilstm_model.summary()

In [ ]:
cnn_bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=5, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

In [ ]:
start_time = time.time()
cnn_bilstm_history = cnn_bilstm_model.fit(
    train_seq, y_train,
    validation_data=(val_seq, y_val),
    epochs=20, 
    batch_size=128,  
    class_weight=class_weights,
    callbacks=cnn_bilstm_callbacks,
    verbose=1
)
cnn_bilstm_time = time.time() - start_time

In [ ]:
cnn_bilstm_pred_prob = cnn_bilstm_model.predict(test_seq, verbose=0)
cnn_bilstm_pred = (cnn_bilstm_pred_prob > 0.5).astype(int).flatten()
cnn_bilstm_results = evaluate_model(y_test, cnn_bilstm_pred, cnn_bilstm_pred_prob, "CNN-BILSTM")
cnn_bilstm_results['training_time'] = cnn_bilstm_time

3-Logistic Regression

In [ ]:
start_time = time.time()

logistic_model = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)

In [ ]:
logistic_model.fit(X_train_tfidf, y_train)
logistic_time = time.time() - start_time

In [ ]:
logistic_pred = logistic_model.predict(X_test_tfidf)
logistic_pred_prob = logistic_model.predict_proba(X_test_tfidf)[:, 1]
logistic_results = evaluate_model(y_test, logistic_pred, logistic_pred_prob, "Logistic Regression")
logistic_results['training_time'] = logistic_time

4-Random Forest

In [ ]:
start_time = time.time()

rf_model = RandomForestClassifier( n_estimators=100, max_depth=None, min_samples_split=5, min_samples_leaf=2, class_weight='balanced_subsample', random_state=42, n_jobs=-1, verbose=0)

In [ ]:
rf_model.fit(X_train_tfidf, y_train)
rf_time = time.time() - start_time

In [ ]:
rf_pred = rf_model.predict(X_test_tfidf)
rf_pred_prob = rf_model.predict_proba(X_test_tfidf)[:, 1]
rf_results = evaluate_model(y_test, rf_pred, rf_pred_prob, "Random Forest")
rf_results['training_time'] = rf_time

5-AraBERT

In [13]:
sample_size = 50000
train_indices = np.random.choice(len(X_train), sample_size, replace=False)
X_train_arabert = X_train[train_indices]
y_train_arabert = y_train[train_indices]
print(f"Training with {len(X_train_arabert):,} samples (subset due to memory constraints)")

Training with 50,000 samples (subset due to memory constraints)


In [14]:
bert_tokenizer = AutoTokenizer.from_pretrained('UBC-NLP/ARBERT')

tokenizer_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [15]:
def prepare_data(texts, max_len=512):
    encodings = bert_tokenizer(
        texts.tolist() if isinstance(texts, np.ndarray) else texts,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors='tf'
    )
    return encodings

train_encodings = prepare_data(X_train_arabert)
val_encodings = prepare_data(X_val)
test_encodings = prepare_data(X_test)

In [16]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

In [17]:
bert_model = TFAutoModelForSequenceClassification.from_pretrained(
    'UBC-NLP/ARBERT', 
    num_labels=2
)

tf_model.h5:   0%|          | 0.00/652M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/ARBERT and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
optimizer = AdamWeightDecay(
    learning_rate=0.00005,
    weight_decay_rate=0.01,
    epsilon=1e-08,
    exclude_from_weight_decay=["LayerNorm", "bias"]
)

In [20]:
bert_model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [21]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
start_time = time.time()

history = bert_model.fit(
    train_encodings.data,
    y_train_arabert,
    validation_data=(val_encodings.data, y_val),
    epochs=10,
    callbacks=callbacks,
    batch_size=64,
    verbose=1
)

arabert_time = time.time() - start_time

Epoch 1/10


In [ ]:
test_loss, test_acc = bert_model.evaluate(test_encodings.data, y_test, verbose=0)
print(f"AraBERT Test Accuracy: {test_acc:.4f}")
print(f"AraBERT Test Loss: {test_loss:.4f}")

In [ ]:
predictions = bert_model.predict(test_encodings.data, verbose=0)
y_pred_arabert = np.argmax(predictions.logits, axis=1)
y_pred_probs = tf.nn.softmax(predictions.logits).numpy()

In [ ]:
arabert_results = evaluate_model(y_test, y_pred_arabert, y_pred_probs[:, 1], "AraBERT (TF/Keras)")
arabert_results['training_time'] = arabert_time

6-Arabic DistilBERT

In [ ]:
 distilbert_tokenizer = AutoTokenizer.from_pretrained('asafaya/distilbert-base-arabic')

In [ ]:
def prepare_distilbert_data(texts, max_len=128):
        encodings = distilbert_tokenizer(
            texts.tolist() if isinstance(texts, np.ndarray) else texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='tf'
        )
        return encodings

train_distilbert_encodings = prepare_distilbert_data(X_train)
val_distilbert_encodings = prepare_distilbert_data(X_val)
test_distilbert_encodings = prepare_distilbert_data(X_test)

In [ ]:
distilbert_model = TFAutoModelForSequenceClassification.from_pretrained('asafaya/distilbert-base-arabic', num_labels=2)

In [ ]:
distilbert_optimizer = AdamWeightDecay(learning_rate=0.00005, weight_decay_rate=0.01, epsilon=1e-08, exclude_from_weight_decay=["LayerNorm", "bias"])

In [ ]:
distilbert_model.compile(
        optimizer=distilbert_optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

In [ ]:
distilbert_callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            'best_distilbert_model',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

In [ ]:
distilbert_start_time = time.time()

distilbert_history = distilbert_model.fit(
        train_distilbert_encodings.data,
        y_train,
        validation_data=(val_distilbert_encodings.data, y_val),
        epochs=10,
        batch_size=32,
        callbacks=distilbert_callbacks,
        verbose=1)

distilbert_time = time.time() - distilbert_start_time

In [ ]:
distilbert_test_loss, distilbert_test_acc = distilbert_model.evaluate(
        test_distilbert_encodings.data,
        y_test,
        verbose=0
    )
print(f"دقة Arabic DistilBERT على بيانات الاختبار: {distilbert_test_acc:.4f}")
print(f"خسارة Arabic DistilBERT على بيانات الاختبار: {distilbert_test_loss:.4f}")

In [ ]:
distilbert_predictions = distilbert_model.predict(test_distilbert_encodings.data, verbose=0)
y_pred_distilbert = np.argmax(distilbert_predictions.logits, axis=1)
y_pred_distilbert_probs = tf.nn.softmax(distilbert_predictions.logits).numpy()

In [ ]:
distilbert_results = evaluate_model(
        y_test,
        y_pred_distilbert,
        y_pred_distilbert_probs[:, 1],
        "Arabic DistilBERT"
    )

In [ ]:
distilbert_results['training_time'] = distilbert_time
distilbert_results['data_size'] = len(X_train)

7-AraBERTv2

In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"

In [ ]:
arabertv2_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def prepare_arabertv2_data(texts, max_len=128):
        encodings = arabertv2_tokenizer(
            texts.tolist() if isinstance(texts, np.ndarray) else texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='tf'
        )
        return encodings

train_arabertv2_encodings = prepare_arabertv2_data(X_train)
val_arabertv2_encodings = prepare_arabertv2_data(X_val)
test_arabertv2_encodings = prepare_arabertv2_data(X_test)

In [ ]:
arabertv2_model = TFAutoModelForSequenceClassification.from_pretrained( MODEL_NAME, num_labels=2)

In [ ]:
arabertv2_optimizer = AdamWeightDecay(
        learning_rate=3e-5,
        weight_decay_rate=0.01,
        epsilon=1e-08,
        exclude_from_weight_decay=["LayerNorm", "bias"]
    )

In [ ]:
arabertv2_model.compile(
        optimizer=arabertv2_optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

In [ ]:
arabertv2_callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            'best_arabertv2_model',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

In [ ]:
arabertv2_start_time = time.time()

arabertv2_history = arabertv2_model.fit(
        train_arabertv2_encodings.data,
        y_train,
        validation_data=(val_arabertv2_encodings.data, y_val),
        epochs=3,
        batch_size=16,
        callbacks=arabertv2_callbacks,
        verbose=1
    )

arabertv2_time = time.time() - arabertv2_start_time

In [ ]:
 arabertv2_test_loss, arabertv2_test_acc = arabertv2_model.evaluate(
        test_arabertv2_encodings.data,
        y_test,
        verbose=0
    )
print(f"دقة AraBERTv2 على الاختبار: {arabertv2_test_acc:.4f}")
print(f"خسارة AraBERTv2 على الاختبار: {arabertv2_test_loss:.4f}")

In [ ]:
arabertv2_predictions = arabertv2_model.predict(test_arabertv2_encodings.data, verbose=0)
y_pred_arabertv2 = np.argmax(arabertv2_predictions.logits, axis=1)
y_pred_arabertv2_probs = tf.nn.softmax(arabertv2_predictions.logits).numpy()

In [ ]:
arabertv2_results = evaluate_model(
        y_test,
        y_pred_arabertv2,
        y_pred_arabertv2_probs[:, 1],
        "AraBERTv2 (TF/Keras)"
    )
arabertv2_results['training_time'] = arabertv2_time
arabertv2_results['data_size'] = len(X_train)

In [ ]:
arabertv2_model.save_pretrained('arabertv2_tf_model')
arabertv2_tokenizer.save_pretrained('arabertv2_tf_model')
print("Model AraBERTv2 'arabertv2_tf_model'")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(arabertv2_history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(arabertv2_history.history['val_accuracy'], label='Validation Accuracy', color='red')
plt.title('AraBERTv2 Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(arabertv2_history.history['loss'], label='Training Loss', color='blue')
plt.plot(arabertv2_history.history['val_loss'], label='Validation Loss', color='red')
plt.title('AraBERTv2 Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

**10-Compare all models & Visualization**

In [ ]:
all_results = [
    bilstm_results,
    cnn_bilstm_results,
    logistic_results,
    rf_results
]

if arabert_results is not None:
    all_results.append(arabert_results)

if distilbert_results is not None:
    all_results.append(distilbert_results)

if arabertv2_results is not None:
    all_results.append(arabertv2_results)

results_df = pd.DataFrame(all_results)
results_df = results_df[['model_name', 'accuracy', 'precision', 'recall', 'f1', 'training_time']]

print("\n" + "="*60)
print("Detailed table (all links)")
print("="*60)
print(results_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(14, 8))

In [ ]:
plt.subplot(2, 2, 1)
models = results_df['model_name']
accuracies = results_df['accuracy']
bars = plt.bar(models, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'][:len(models)])
plt.title('Compare the accuracy of models', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.xticks(rotation=45, ha='right')
plt.ylim([0, 1])
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.3f}',
             ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(2, 2, 2)
f1_scores = results_df['f1']
bars = plt.bar(models, f1_scores, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'][:len(models)])
plt.title('Compare F1-Score', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('F1-Score')
plt.xticks(rotation=45, ha='right')
plt.ylim([0, 1])
for bar, f1 in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{f1:.3f}',
             ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(2, 2, 3)
training_times = results_df['training_time']
bars = plt.bar(models, training_times, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'][:len(models)])
plt.title('Training time (in seconds)', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('(Time (seconds)')
plt.xticks(rotation=45, ha='right')
for bar, time_val in zip(bars, training_times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{time_val:.1f}',
             ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(2, 2, 4)
x = np.arange(len(models))
width = 0.35
precision_vals = results_df['precision']
recall_vals = results_df['recall']

plt.bar(x - width/2, precision_vals, width, label='Precision', color='#2ca02c')
plt.bar(x + width/2, recall_vals, width, label='Recall', color='#d62728')

plt.title('Compare Precision و Recall', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Value')
plt.xticks(x, models, rotation=45, ha='right')
plt.legend()
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

**11-Save the best model**

In [ ]:
best_model_idx = results_df['f1'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'model_name']
best_model_f1 = results_df.loc[best_model_idx, 'f1']

print(f"\n🏆 Best model: {best_model_name} (F1-Score: {best_model_f1:.4f})")

if best_model_name == "BiLSTM":
    bilstm_model.save("best_model_bilstm.keras")
    print("✅ Best model (BiLSTM) saved as best_model_bilstm.keras")

elif best_model_name == "LSTM with TF-IDF":
    lstm_tfidf_model.save("best_model_lstm_tfidf.keras")
    print("✅ Best model (LSTM with TF-IDF) saved as best_model_lstm_tfidf.keras")

elif best_model_name == "CNN-LSTM":
    cnn_lstm_model.save("best_model_cnn_lstm.keras")
    print("✅ Best model (CNN-LSTM) saved as best_model_cnn_lstm.keras")

elif best_model_name == "Logistic Regression":
    joblib.dump(logistic_model, "best_model_logistic.pkl")
    joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")
    print("✅ Best model (Logistic Regression) saved as best_model_logistic.pkl")

elif best_model_name == "Random Forest":
    joblib.dump(rf_model, "best_model_random_forest.pkl")
    joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")
    print("✅ Best model (Random Forest) saved as best_model_random_forest.pkl")

elif "AraBERT (TF/Keras)" in best_model_name and arabert_results is not None:
    print("✅ AraBERT model saved in 'arabert_tf_model_full' folder")

elif "Arabic DistilBERT" in best_model_name and distilbert_results is not None:
    print("✅ Arabic DistilBERT model saved in 'distilbert_arabic_model' folder")

elif "AraBERTv2 (TF/Keras)" in best_model_name and arabertv2_results is not None:
    print("✅ AraBERTv2 model saved in 'arabertv2_tf_model' folder")
else:
    print("⚠️ Best model was not saved")

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"• Models tested: {len(all_results)}")
print(f"• Best model: {best_model_name}")
print(f"• Best F1-Score: {best_model_f1:.4f}")
print(f"• Dataset size: {len(df2):,} samples")
print(f"• Class distribution: {dict(df2['label'].value_counts())}")
print(f"• Training samples: {len(X_train):,}")
print(f"• Test samples: {len(X_test):,}")

print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)
print(f"1. {best_model_name} is best for balanced performance")
print(f"2. Logistic Regression is fastest for deployment")
print(f"3. Deep learning models (BiLSTM, CNN-LSTM) work well for long texts")
print(f"4. Use class weights to handle imbalanced data")

print("\n" + "="*60)
print("USAGE INSTRUCTIONS")
print("="*60)
print("To predict new Arabic news articles:")
print("1. Load the saved model and artifacts")
print("2. Preprocess text using the same functions")
print("3. Use appropriate vectorization (TF-IDF or tokenization)")
print("4. Make predictions using the loaded model")

print("\n" + "="*60)
print("✅ EXPERIMENT COMPLETED SUCCESSFULLY!")
print("="*60)

In [ ]:
def predict_news_article(text, model_type=None, model_path=None):

    try:
        tokenizer_loaded = joblib.load("tokenizer.pkl")
        label_encoder_loaded = joblib.load("label_encoder.pkl")
        tfidf_loaded = joblib.load("tfidf_vectorizer_backup.pkl")
    except:
        tokenizer_loaded = tokenizer
        label_encoder_loaded = le
        tfidf_loaded = tfidf_vectorizer

    # Preprocess text
    cleaned_text = preprocess_stem(text)

    if model_type == 'logistic' or model_type == 'random_forest':
        # Use TF-IDF
        text_tfidf = tfidf_loaded.transform([cleaned_text])

        if model_type == 'logistic':
            model = joblib.load("best_model_logistic.pkl") if model_path is None else joblib.load(model_path)
            pred = model.predict(text_tfidf)[0]
            prob = model.predict_proba(text_tfidf)[0]
        else:
            model = joblib.load("best_model_random_forest.pkl") if model_path is None else joblib.load(model_path)
            pred = model.predict(text_tfidf)[0]
            prob = model.predict_proba(text_tfidf)[0]

    elif model_type == 'bilstm':
        # Use tokenization
        seq = pad_sequences(tokenizer_loaded.texts_to_sequences([cleaned_text]),
                           maxlen=max_length, padding='post')
        model = tf.keras.models.load_model("best_model_bilstm.keras") if model_path is None else tf.keras.models.load_model(model_path)
        pred_prob = model.predict(seq, verbose=0)[0][0]
        pred = 1 if pred_prob > 0.5 else 0
        prob = [1-pred_prob, pred_prob] if pred == 1 else [pred_prob, 1-pred_prob]

    else:
        # Default to logistic regression
        text_tfidf = tfidf_loaded.transform([cleaned_text])
        model = joblib.load("best_model_logistic.pkl")
        pred = model.predict(text_tfidf)[0]
        prob = model.predict_proba(text_tfidf)[0]

    # Get label
    label = label_encoder_loaded.inverse_transform([pred])[0]
    confidence = max(prob)

    return {
        'text_preview': text[:100] + "..." if len(text) > 100 else text,
        'prediction': label,
        'confidence': float(confidence),
        'credible_probability': float(prob[1] if len(prob) == 2 else pred_prob),
        'not_credible_probability': float(prob[0] if len(prob) == 2 else 1-pred_prob)
    }

# Example usage
if __name__ == "__main__":
    sample_text = "أعلنت وزارة الصحة عن اكتشاف علاج جديد لفيروس كورونا يعطي نتائج مبهرة في التجارب السريرية"
    result = predict_news_article(sample_text, model_type='logistic')
    print("\nExample Prediction:")
    print(f"Text: {result['text_preview']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Credible probability: {result['credible_probability']:.2%}")
    print(f"Not credible probability: {result['not_credible_probability']:.2%}")